# xView3 hard-negative mining

Runs the current models (xview3_train Version 5: VV, VH, 3 seeds) over 20 whole **training** scenes that
have human-checked labels, finds confident detections with no label nearby (false alarms), and cuts
512 px training tiles around them. Retraining with these tiles teaches the model what the sea, coast
and clutter that fooled it look like.

Only manually labelled (xView3 validation) scenes are mined: in AIS-only scenes an "unlabelled
detection" may be a real dark ship, and calling it a negative would teach the model to miss exactly
the ships this project is about.

**Inputs:** `maritime-xview3-mine` dataset + `xview3_train` notebook output (Version 5). GPU on.
**Output:** `hn_<scene>.npz` tile files (same format as the training tiles) for the next training run.

In [ ]:
DETECTOR_SRC = '"""SAR vessel detector: data, model, losses, decoding, xView3-style scoring.\n\nSelf-contained (numpy, scipy, torch, torchvision only) so the same file runs locally,\non the server, and inside a Kaggle notebook (uploaded next to the tiles).\n\nModel: a CenterNet-style point detector. A ResNet encoder (input channels chosen from CHANNELS:\nVV/VH in dB, locally normalised variants, ancillary layers; see make_input)\nwith a small upsampling decoder predicts, at 1/4 resolution (128 x 128 for a 512 tile):\n  heatmap   probability that an object is centred in this cell (focal loss)\n  offset    sub-cell position of the centre (L1)\n  length    object length in metres, as log(1 + m) (L1, where labelled)\n  vessel    logit: vessel vs fixed infrastructure (BCE, where labelled)\n  fishing   logit: fishing vs not, for vessels (BCE, where labelled)\nWhy points, not boxes: xView3 labels are points; a heatmap peak is the natural output\nand is what strong xView3 solutions used.\n\nBaseline: CFAR (constant false alarm rate), the classic radar ship detector: a pixel is a\ndetection if it is much brighter than the sea clutter around it.\n\nScoring follows the xView3 reference metric: per scene, Hungarian matching of predictions\nto labels within 200 m (20 px at 10 m/px); predictions matched to LOW-confidence labels are\ndropped (neither TP nor FP); unmatched LOW labels are not counted as misses.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\nimport numpy as np\n\nPIX_M = 10.0\nMATCH_M = 200.0\nSTRIDE = 4\n\n\n# ---------------------------------------------------------------- data\n\n@dataclass\nclass TileSet:\n    images: np.ndarray        # (n, 2, 512, 512) uint8: VV, VH quantised dB (0 = no data)\n    origins: np.ndarray       # (n, 2) tile origin in scene pixels\n    scene: np.ndarray         # (n,) scene id per tile\n    objects: list             # per tile: structured array of objects inside it\n    ancillary: dict = field(default_factory=dict)    # scene -> {layer: 2-D float array}\n    scene_shape: dict = field(default_factory=dict)  # scene -> (H, W) in SAR pixels\n\n    def __len__(self):\n        return len(self.images)\n\n\ndef load_tiles(paths, keep=None) -> TileSet:\n    """Load tiles from per-scene npz files into ONE preallocated array (no stacking copy).\n    keep(scene_id, n_objects_per_tile) -> bool mask of tiles to keep (default: all)."""\n    paths = sorted(paths)\n    plans = []\n    for p in paths:                                   # pass 1: small arrays only, to size the result\n        z = np.load(p, allow_pickle=False)\n        sid = str(p).rsplit("/", 1)[-1].removesuffix(".npz")\n        org, o = z["origins"], z["objects"]\n        counts = np.bincount(o["tile"], minlength=len(org)) if len(o) else np.zeros(len(org), int)\n        mask = np.ones(len(org), bool) if keep is None else np.asarray(keep(sid, counts), bool)\n        plans.append((p, sid, mask))\n    n = int(sum(m.sum() for _, _, m in plans))\n    images = np.empty((n, 2, 512, 512), np.uint8)\n    origins, scenes, objs, anc, shapes = [], [], [], {}, {}\n    i = 0\n    for p, sid, mask in plans:                        # pass 2: decompress each array once\n        z = np.load(p, allow_pickle=False)\n        im, org, o = z["images"], z["origins"], z["objects"]\n        shapes[sid] = tuple(int(v) for v in z["scene_shape"])\n        anc[sid] = {k[4:]: z[k].astype(np.float32) for k in z.files if k.startswith("anc_")}\n        for t in np.flatnonzero(mask):\n            images[i] = im[t]\n            origins.append(org[t])\n            scenes.append(sid)\n            objs.append(o[o["tile"] == t])\n            i += 1\n        del im\n    return TileSet(images, np.array(origins).reshape(-1, 2), np.array(scenes), objs, anc, shapes)\n\n\n# ---------------------------------------------------------------- input channels\n# Each channel is a float32 512x512 map, roughly in [-3, 3]. Which ones the model sees is a\n# config choice (input-layer selection by ablation).\nCHANNELS = ("VV", "VH", "VV_local", "VH_local", "VV_minus_VH", "bathymetry", "wind_speed",\n            "wind_dir_sin", "wind_dir_cos", "land_mask")\n\n\ndef _db(u8):\n    return np.where(u8 > 0, (u8.astype(np.float32) - 1) / 254 * 60.0 - 50.0, np.nan)\n\n\ndef _anc_crop(layer, origin, scene_shape, size=512):\n    """Crop a coarse scene-wide ancillary layer to one tile and upsample it to size x size."""\n    from scipy import ndimage\n    H, W = scene_shape\n    sy, sx = layer.shape[0] / H, layer.shape[1] / W           # ancillary px per SAR px\n    r0, c0 = origin[0] * sy, origin[1] * sx\n    ys = np.clip(r0 + (np.arange(size) + 0.5) * sy - 0.5, 0, layer.shape[0] - 1)\n    xs = np.clip(c0 + (np.arange(size) + 0.5) * sx - 0.5, 0, layer.shape[1] - 1)\n    filled = np.where(np.isnan(layer), np.nanmedian(layer) if np.isfinite(layer).any() else 0, layer)\n    yy, xx = np.meshgrid(ys, xs, indexing="ij")\n    return ndimage.map_coordinates(filled, [yy, xx], order=1, mode="nearest").astype(np.float32)\n\n\ndef make_input(img_u8, channels, anc=None, origin=(0, 0), scene_shape=(512, 512)):\n    """uint8 (2, H, W) tile (+ scene ancillary layers) -> float32 (len(channels), H, W)."""\n    vv, vh = _db(img_u8[0]), _db(img_u8[1])\n    anc = anc or {}\n    out = []\n    for ch in channels:\n        if ch == "VV":\n            a = (vv + 20.0) / 8.0                       # global scaling (sea ~ -20 dB)\n        elif ch == "VH":\n            a = (vh + 28.0) / 8.0\n        elif ch == "VV_local":                          # subtract this tile\'s sea level (EDA Q2)\n            a = (vv - np.nanmedian(vv)) / 8.0 if np.isfinite(vv).any() else vv\n        elif ch == "VH_local":\n            a = (vh - np.nanmedian(vh)) / 8.0 if np.isfinite(vh).any() else vh\n        elif ch == "VV_minus_VH":\n            a = (vv - vh - 8.0) / 6.0\n        elif ch in ("bathymetry", "wind_speed", "wind_dir_sin", "wind_dir_cos", "land_mask"):\n            key = {"bathymetry": "bathymetry", "wind_speed": "owiWindSpeed", "land_mask": "owiMask",\n                   "wind_dir_sin": "owiWindDirection", "wind_dir_cos": "owiWindDirection"}[ch]\n            if key not in anc:\n                a = np.zeros(vv.shape, np.float32)      # layer absent for this scene\n            else:\n                m = _anc_crop(anc[key], origin, scene_shape, vv.shape[-1])\n                if ch == "bathymetry":                  # metres, negative = below sea level\n                    a = -np.sign(m) * np.log1p(np.abs(m)) / 4.0\n                elif ch == "wind_speed":\n                    a = (m - 7.0) / 4.0\n                elif ch == "wind_dir_sin":\n                    a = np.sin(np.deg2rad(m))\n                elif ch == "wind_dir_cos":\n                    a = np.cos(np.deg2rad(m))\n                else:\n                    a = (m > 0).astype(np.float32)      # owiMask: 0 = ocean\n        else:\n            raise ValueError(f"unknown channel {ch}; choose from {CHANNELS}")\n        out.append(np.nan_to_num(a, nan=0.0))\n    return np.stack(out).astype(np.float32)\n\n\ndef render_targets(objs, size=512, stride=STRIDE, sigma=1.5):\n    """Heatmap + per-centre regression targets at 1/stride resolution."""\n    h = size // stride\n    heat = np.zeros((h, h), np.float32)\n    ys, xs = np.mgrid[0:h, 0:h]\n    centres = []\n    for o in objs:\n        cy, cx = o["row"] / stride, o["col"] / stride\n        iy, ix = int(cy), int(cx)\n        if not (0 <= iy < h and 0 <= ix < h):\n            continue\n        heat = np.maximum(heat, np.exp(-((ys - cy) ** 2 + (xs - cx) ** 2) / (2 * sigma**2)))\n        heat[iy, ix] = 1.0\n        centres.append((iy, ix, cy - iy, cx - ix, o["length_m"], o["is_vessel"], o["is_fishing"]))\n    return heat, np.array(centres, np.float32).reshape(-1, 7)\n\n\ndef augment(img, objs, rng):\n    """Random flips and 90-degree rotations (radar statistics are unchanged by these).\n    img: (C, H, W) array of any dtype; objs: structured array with row/col."""\n    size = img.shape[-1]\n    rows, cols = objs["row"].astype(np.float64), objs["col"].astype(np.float64)\n    if rng.random() < 0.5:\n        img, cols = img[:, :, ::-1], size - 1 - cols\n    if rng.random() < 0.5:\n        img, rows = img[:, ::-1, :], size - 1 - rows\n    k = int(rng.integers(4))\n    for _ in range(k):                       # rotate 90 deg counter-clockwise\n        img = np.rot90(img, 1, axes=(1, 2))\n        rows, cols = size - 1 - cols, rows\n    out = objs.copy()\n    out["row"], out["col"] = rows, cols\n    return np.ascontiguousarray(img), out\n\n\n# ---------------------------------------------------------------- model\n\ndef build_model(backbone: str = "resnet18", pretrained: bool = False, in_ch: int = 2):\n    import torch\n    import torch.nn as nn\n    import torchvision\n\n    weights = "DEFAULT" if pretrained else None\n    enc = getattr(torchvision.models, backbone)(weights=weights)\n    w = enc.conv1.weight.data                      # (64, 3, 7, 7): adapt RGB -> in_ch channels\n    enc.conv1 = nn.Conv2d(in_ch, 64, 7, 2, 3, bias=False)\n    enc.conv1.weight.data = w.mean(1, keepdim=True).repeat(1, in_ch, 1, 1) * (3.0 / in_ch)\n    ch = [64, 64, 128, 256, 512] if backbone in ("resnet18", "resnet34") else [64, 256, 512, 1024, 2048]\n\n    class Net(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu)        # 1/2\n            self.pool = enc.maxpool                                        # 1/4\n            self.l1, self.l2, self.l3, self.l4 = enc.layer1, enc.layer2, enc.layer3, enc.layer4\n            def up(cin, cout):\n                return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())\n            self.u3 = up(ch[4] + ch[3], 256)\n            self.u2 = up(256 + ch[2], 128)\n            self.u1 = up(128 + ch[1], 64)\n            def head(cout, bias=0.0):\n                h = nn.Sequential(nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.Conv2d(64, cout, 1))\n                nn.init.constant_(h[-1].bias, bias)\n                return h\n            self.heat = head(1, -2.19)        # prior p ~ 0.1, standard CenterNet init\n            self.offset, self.length = head(2), head(1)\n            self.vessel, self.fishing = head(1), head(1)\n\n        def forward(self, x):\n            f1 = self.l1(self.pool(self.stem(x)))          # 1/4\n            f2 = self.l2(f1)                                # 1/8\n            f3 = self.l3(f2)                                # 1/16\n            f4 = self.l4(f3)                                # 1/32\n            def cat_up(a, b):\n                return torch.cat([nn.functional.interpolate(a, size=b.shape[-2:], mode="nearest"), b], 1)\n            d = self.u3(cat_up(f4, f3))\n            d = self.u2(cat_up(d, f2))\n            d = self.u1(cat_up(d, f1))                      # back to 1/4\n            return {"heat": self.heat(d), "offset": self.offset(d), "length": self.length(d),\n                    "vessel": self.vessel(d), "fishing": self.fishing(d)}\n\n    return Net()\n\n\ndef focal_loss(logits, target, alpha=2.0, beta=4.0):\n    """CenterNet focal loss: down-weights easy negatives and negatives near a centre."""\n    import torch\n    p = torch.sigmoid(logits).clamp(1e-4, 1 - 1e-4)\n    pos = target.eq(1).float()\n    neg = 1 - pos\n    pos_loss = -torch.log(p) * (1 - p) ** alpha * pos\n    neg_loss = -torch.log(1 - p) * p**alpha * (1 - target) ** beta * neg\n    n = pos.sum().clamp(min=1)\n    return (pos_loss.sum() + neg_loss.sum()) / n\n\n\ndef compute_loss(out, heat_t, centres_list):\n    """centres_list: per image (k, 7) array [iy, ix, dy, dx, length, vessel, fishing]."""\n    import torch\n    import torch.nn.functional as F\n    loss_heat = focal_loss(out["heat"][:, 0], heat_t)\n    off_l, len_l, ves_l, fis_l = [], [], [], []\n    for b, c in enumerate(centres_list):\n        if len(c) == 0:\n            continue\n        c = torch.as_tensor(c, device=heat_t.device)\n        iy, ix = c[:, 0].long(), c[:, 1].long()\n        off_l.append(F.l1_loss(out["offset"][b, :, iy, ix].T, c[:, 2:4], reduction="sum"))\n        m = ~torch.isnan(c[:, 4])\n        if m.any():\n            len_l.append(F.l1_loss(out["length"][b, 0, iy, ix][m], torch.log1p(c[m, 4]), reduction="sum"))\n        m = ~torch.isnan(c[:, 5])\n        if m.any():\n            ves_l.append(F.binary_cross_entropy_with_logits(out["vessel"][b, 0, iy, ix][m], c[m, 5], reduction="sum"))\n        m = ~torch.isnan(c[:, 6]) & (c[:, 5] == 1)\n        if m.any():\n            fis_l.append(F.binary_cross_entropy_with_logits(out["fishing"][b, 0, iy, ix][m], c[m, 6], reduction="sum"))\n    n = max(sum(len(c) for c in centres_list), 1)\n    def total(lst):\n        return torch.stack(lst).sum() / n if lst else heat_t.new_zeros(())\n    parts = {"heat": loss_heat, "offset": total(off_l), "length": total(len_l),\n             "vessel": total(ves_l), "fishing": total(fis_l)}\n    loss = parts["heat"] + parts["offset"] + 0.5 * parts["length"] + 0.5 * parts["vessel"] + 0.5 * parts["fishing"]\n    return loss, {k: float(v.detach()) for k, v in parts.items()}\n\n\ndef decode(out, threshold=0.3, max_det=100):\n    """Heatmap peaks (3x3 local maxima above threshold) -> detections per image:\n    (row, col, score, length_m, p_vessel, p_fishing) in tile pixel coordinates."""\n    import torch\n    import torch.nn.functional as F\n    heat = torch.sigmoid(out["heat"][:, 0])\n    peaks = heat * (heat == F.max_pool2d(heat[:, None], 3, 1, 1)[:, 0])\n    results = []\n    for b in range(heat.shape[0]):\n        scores, idx = peaks[b].flatten().topk(min(max_det, peaks[b].numel()))\n        keep = scores > threshold\n        scores, idx = scores[keep], idx[keep]\n        w = heat.shape[-1]\n        iy, ix = idx // w, idx % w\n        off = out["offset"][b, :, iy, ix]\n        rows = (iy.float() + off[0]) * STRIDE\n        cols = (ix.float() + off[1]) * STRIDE\n        length = torch.expm1(out["length"][b, 0, iy, ix]).clamp(min=0)\n        pv = torch.sigmoid(out["vessel"][b, 0, iy, ix])\n        pf = torch.sigmoid(out["fishing"][b, 0, iy, ix])\n        results.append(torch.stack([rows, cols, scores, length, pv, pf], 1).detach().cpu().numpy())\n    return results\n\n\n# ---------------------------------------------------------------- CFAR baseline\n\ndef cfar(img_u8, guard=4, train=16, k=5.0, min_px=2):\n    """Cell-averaging CFAR on VV (dB): detect pixels > local clutter mean + k * std,\n    with clutter measured in a ring outside a guard window; returns (row, col, score)."""\n    from scipy import ndimage\n    db = img_u8[0].astype(np.float32)\n    valid = db > 0\n    big, small = 2 * (guard + train) + 1, 2 * guard + 1\n    def box(a, s):\n        return ndimage.uniform_filter(a, s, mode="nearest") * s * s\n    n = big * big - small * small\n    s1 = box(db, big) - box(db, small)\n    s2 = box(db**2, big) - box(db**2, small)\n    mean = s1 / n\n    std = np.sqrt(np.maximum(s2 / n - mean**2, 1e-6))\n    det = valid & (db > mean + k * std)\n    lab, nlab = ndimage.label(det)\n    if nlab == 0:\n        return np.zeros((0, 3))\n    idx = np.arange(1, nlab + 1)\n    sizes = ndimage.sum(det, lab, idx)\n    com = np.array(ndimage.center_of_mass(det, lab, idx)).reshape(-1, 2)\n    peak = ndimage.maximum((db - mean) / std, lab, idx)\n    keep = sizes >= min_px\n    return np.column_stack([com[keep], peak[keep]])\n\n\n# ---------------------------------------------------------------- xView3-style scoring\n\ndef match(pred_rc, gt_rc, max_px=MATCH_M / PIX_M):\n    """Hungarian matching within max_px; returns list of (pred_idx, gt_idx)."""\n    from scipy.optimize import linear_sum_assignment\n    from scipy.spatial import distance_matrix\n    if len(pred_rc) == 0 or len(gt_rc) == 0:\n        return []\n    d = distance_matrix(pred_rc, gt_rc)\n    big = 1e9\n    rows, cols = linear_sum_assignment(np.where(d <= max_px, d, big))\n    return [(r, c) for r, c in zip(rows, cols, strict=True) if d[r, c] <= max_px]\n\n\ndef score(preds: dict, gts: dict) -> dict:\n    """preds/gts: scene_id -> structured arrays.\n    preds: rows/cols (scene px), is_vessel, is_fishing, length_m (vessel/fishing as 0/1).\n    gts: same + confidence. Returns detection P/R/F1, vessel F1, fishing F1, length error."""\n    tp = fp = fn = 0\n    vessel, fishing, length_err = [], [], []\n    for sid in set(preds) | set(gts):\n        p = preds.get(sid, np.zeros(0, dtype=[("row", "f4"), ("col", "f4")]))\n        g = gts.get(sid, np.zeros(0, dtype=[("row", "f4"), ("col", "f4"), ("confidence", "U6")]))\n        pairs = match(np.column_stack([p["row"], p["col"]]) if len(p) else np.zeros((0, 2)),\n                      np.column_stack([g["row"], g["col"]]) if len(g) else np.zeros((0, 2)))\n        low = (g["confidence"] == "LOW") if len(g) else np.zeros(0, bool)\n        matched_p = {a for a, _ in pairs}\n        matched_g = {b for _, b in pairs}\n        good = [(a, b) for a, b in pairs if not low[b]]\n        tp += len(good)\n        fp += len(p) - len(matched_p)                     # unmatched predictions\n        fn += int(sum(1 for b in range(len(g)) if b not in matched_g and not low[b]))\n        for a, b in good:\n            if "is_vessel" in g.dtype.names and not np.isnan(g["is_vessel"][b]):\n                vessel.append((p["is_vessel"][a], g["is_vessel"][b]))\n                if g["is_vessel"][b] == 1 and not np.isnan(g["is_fishing"][b]):\n                    fishing.append((p["is_fishing"][a], g["is_fishing"][b]))\n            if "length_m" in g.dtype.names and not np.isnan(g["length_m"][b]):\n                length_err.append(abs(p["length_m"][a] - g["length_m"][b]) / max(g["length_m"][b], 1e-6))\n\n    def f1(tp_, fp_, fn_):\n        prec = tp_ / (tp_ + fp_) if tp_ + fp_ else 0.0\n        rec = tp_ / (tp_ + fn_) if tp_ + fn_ else 0.0\n        return prec, rec, (2 * prec * rec / (prec + rec) if prec + rec else 0.0)\n\n    def class_f1(pairs):\n        if not pairs:\n            return float("nan")\n        a = np.array(pairs, dtype=float)\n        t = ((a[:, 0] >= 0.5) & (a[:, 1] == 1)).sum()\n        f_p = ((a[:, 0] >= 0.5) & (a[:, 1] == 0)).sum()\n        f_n = ((a[:, 0] < 0.5) & (a[:, 1] == 1)).sum()\n        return f1(t, f_p, f_n)[2]\n\n    prec, rec, f1d = f1(tp, fp, fn)\n    pel = 1.0 - min(float(np.mean(length_err)), 1.0) if length_err else float("nan")\n    return {"tp": tp, "fp": fp, "fn": fn, "precision": prec, "recall": rec, "f1_detect": f1d,\n            "f1_vessel": class_f1(vessel), "f1_fishing": class_f1(fishing), "length_score": pel}\n\n\n# ---------------------------------------------------------------- whole-scene inference + full xView3 metric\n\ndef scene_tiles(H, W, size=512, overlap=64):\n    """Tile origins covering an H x W scene with `overlap` px between neighbours (last row/col flush\n    with the scene edge)."""\n    def starts(n):\n        if n <= size:\n            return [0]\n        s = list(range(0, n - size, size - overlap)) + [n - size]\n        return sorted(set(s))\n    return [(r, c) for r in starts(H) for c in starts(W)]\n\n\ndef keep_core(dets, origin, scene_shape, size=512, margin=32):\n    """Drop detections within `margin` px of a tile edge, unless that edge is the scene edge\n    (a neighbouring tile sees those objects with more context)."""\n    if len(dets) == 0:\n        return dets\n    r0, c0 = origin\n    H, W = scene_shape\n    r, c = dets[:, 0], dets[:, 1]\n    lo_r = 0 if r0 == 0 else margin\n    lo_c = 0 if c0 == 0 else margin\n    hi_r = size if r0 + size >= H else size - margin\n    hi_c = size if c0 + size >= W else size - margin\n    return dets[(r >= lo_r) & (r < hi_r) & (c >= lo_c) & (c < hi_c)]\n\n\ndef merge_close(dets, radius_px=10.0):\n    """Greedy de-duplication: keep the highest-score detection, drop others within radius_px.\n    dets: (n, k) with columns row, col, score, ..."""\n    if len(dets) < 2:\n        return dets\n    from scipy.spatial import cKDTree\n    order = np.argsort(-dets[:, 2])\n    d = dets[order]\n    tree = cKDTree(d[:, :2])\n    alive = np.ones(len(d), bool)\n    for i in range(len(d)):\n        if alive[i]:\n            nb = tree.query_ball_point(d[i, :2], radius_px)\n            alive[[j for j in nb if j > i]] = False\n    return d[alive]\n\n\ndef shore_distance_px(points_rc, contours):\n    """Distance (px) from each point to the nearest shoreline vertex; inf when there is no shoreline."""\n    pts = np.asarray(points_rc, float).reshape(-1, 2)\n    segs = [np.asarray(c, float).reshape(-1, 2) for c in (contours if contours is not None else [])]\n    segs = [s for s in segs if len(s)]\n    if not segs or not len(pts):\n        return np.full(len(pts), np.inf)\n    from scipy.spatial import cKDTree\n    return cKDTree(np.concatenate(segs)).query(pts)[0]\n\n\ndef xview3_metrics(preds: dict, gts: dict, shorelines: dict | None = None, shore_km=2.0) -> dict:\n    """Full xView3 scoring: score() plus close-to-shore detection F1 (labels and predictions within\n    shore_km of the shoreline, matched among themselves) and the aggregate\n    F1_D * (1 + F1_shore + F1_vessel + F1_fishing + length_score) / 5 (NaN parts count as 0)."""\n    out = score(preds, gts)\n    if shorelines:\n        lim = shore_km * 1000 / PIX_M\n        def near(d, sid):\n            if d is None or not len(d):\n                return d\n            dist = shore_distance_px(np.column_stack([d["row"], d["col"]]), shorelines.get(sid))\n            return d[dist <= lim]\n        sids = [s for s in set(preds) | set(gts) if s in shorelines]\n        ps = {s: near(preds.get(s), s) for s in sids if preds.get(s) is not None}\n        gs = {s: near(gts.get(s), s) for s in sids if gts.get(s) is not None}\n        sh = score(ps, gs)\n        out.update(f1_shore=sh["f1_detect"], shore_tp=sh["tp"], shore_fp=sh["fp"], shore_fn=sh["fn"])\n    else:\n        out.update(f1_shore=float("nan"))\n    def z(v):\n        return 0.0 if v is None or (isinstance(v, float) and np.isnan(v)) else float(v)\n    out["aggregate"] = out["f1_detect"] * (1 + z(out["f1_shore"]) + z(out["f1_vessel"])\n                                           + z(out["f1_fishing"]) + z(out["length_score"])) / 5\n    return out\n\n\n# ---------------------------------------------------------------- shared scene pipeline (eval, mining, calibration)\n\n# Same layout as maritime.sar.xview3.OBJECT_DTYPE (kept here so this file stays self-contained on Kaggle).\nOBJECT_DTYPE = np.dtype([\n    ("tile", "i4"), ("row", "i2"), ("col", "i2"),\n    ("is_vessel", "f4"), ("is_fishing", "f4"), ("length_m", "f4"),\n    ("confidence", "U6"), ("shore_km", "f4"),\n])\n\n\ndef predict_scene(models, img, channels=("VV", "VH"), size=512, overlap=64, batch=32, min_valid=0.05,\n                  device="cpu", num_workers=0, merge_px=4.0, decode_threshold=0.05, max_det=200):\n    """Slide `models` over a whole (2, H, W) uint8 scene. Returns (dets, origins): dets is a list with one\n    (n, 6) array per model plus a final one for the output-averaged ensemble; columns are\n    row, col, score, length_m, p_vessel, p_fishing in scene pixels."""\n    import torch\n    H, W = img.shape[1:]\n    origins = [o for o in scene_tiles(H, W, size, overlap)\n               if (img[0, o[0]:o[0] + size:8, o[1]:o[1] + size:8] > 0).mean() >= min_valid]\n\n    class _Tiles(torch.utils.data.Dataset):\n        def __len__(self):\n            return len(origins)\n\n        def __getitem__(self, i):\n            r, c = origins[i]\n            t = img[:, r:r + size, c:c + size]\n            if t.shape[1:] != (size, size):\n                t = np.pad(t, ((0, 0), (0, size - t.shape[1]), (0, size - t.shape[2])))\n            return torch.from_numpy(make_input(t, list(channels))), i\n\n    dl = torch.utils.data.DataLoader(_Tiles(), batch_size=batch, num_workers=num_workers)\n    per = [[] for _ in range(len(models) + 1)]\n    with torch.no_grad():\n        for x, idx in dl:\n            x = x.to(device)\n            with torch.autocast("cuda" if str(device).startswith("cuda") else "cpu",\n                                enabled=str(device).startswith("cuda")):\n                outs = [{k: v.float() for k, v in m(x).items()} for m in models]\n            outs.append({k: torch.stack([o[k] for o in outs]).mean(0) for k in outs[0]})\n            for j, o in enumerate(outs):\n                for d, i in zip(decode(o, threshold=decode_threshold, max_det=max_det), idx.tolist(), strict=True):\n                    d = keep_core(d, origins[i], (H, W), size, overlap // 2)\n                    if len(d):\n                        d = d.copy()\n                        d[:, 0] += origins[i][0]\n                        d[:, 1] += origins[i][1]\n                        per[j].append(d)\n    dets = [merge_close(np.concatenate(p), merge_px) if p else np.zeros((0, 6)) for p in per]\n    return dets, origins\n\n\ndef unmatched_detections(dets, label_rc, min_score=0.3, radius_px=30.0):\n    """Detections scoring >= min_score with no label of ANY confidence within radius_px:\n    candidate false alarms (a lenient radius so slightly misplaced labels don\'t count as negatives)."""\n    d = dets[dets[:, 2] >= min_score]\n    if not len(d) or not len(label_rc):\n        return d\n    from scipy.spatial import cKDTree\n    dist = cKDTree(np.asarray(label_rc, float).reshape(-1, 2)).query(d[:, :2])[0]\n    return d[dist > radius_px]\n\n\ndef hard_negative_tiles(img, false_alarms, labels, max_tiles=150, size=512, margin=64, seed=0):\n    """Training tiles around false alarms: each false alarm lands at a random spot at least `margin` px\n    inside its tile; false alarms already inside an earlier tile are skipped (no near-duplicate tiles). Any real labels inside a\n    tile are kept as objects, so the tile teaches "this is sea" without erasing true ships.\n    labels: structured array with row, col, is_vessel, is_fishing, length_m, confidence, shore_km (scene px).\n    Returns images (n, 2, size, size) uint8, origins (n, 2), objects (OBJECT_DTYPE)."""\n    rng = np.random.default_rng(seed)\n    H, W = img.shape[1:]\n    fa = false_alarms[np.argsort(-false_alarms[:, 2])] if len(false_alarms) else false_alarms\n    origins = []\n    for r, c in fa[:, :2] if len(fa) else []:\n        if len(origins) >= max_tiles:\n            break\n        if any(o[0] + 16 <= r < o[0] + size - 16 and o[1] + 16 <= c < o[1] + size - 16\n               for o in origins):                  # already inside a tile: avoid near-duplicate tiles\n            continue\n        r0 = int(np.clip(r - rng.integers(margin, size - margin), 0, max(H - size, 0)))\n        c0 = int(np.clip(c - rng.integers(margin, size - margin), 0, max(W - size, 0)))\n        origins.append((r0, c0))\n    images = np.zeros((len(origins), 2, size, size), np.uint8)\n    objs = []\n    for t, (r0, c0) in enumerate(origins):\n        tile = img[:, r0:r0 + size, c0:c0 + size]\n        images[t, :, :tile.shape[1], :tile.shape[2]] = tile\n        if len(labels):\n            m = (labels["row"] >= r0) & (labels["row"] < r0 + size) & (labels["col"] >= c0) & (labels["col"] < c0 + size)\n            for lb in labels[m]:\n                objs.append((t, int(lb["row"]) - r0, int(lb["col"]) - c0, lb["is_vessel"], lb["is_fishing"],\n                             lb["length_m"], lb["confidence"], lb["shore_km"]))\n    return images, np.array(origins, np.int32).reshape(-1, 2), np.array(objs, OBJECT_DTYPE)\n'
print(f'embedded detector code: {len(DETECTOR_SRC):,} chars')

In [ ]:
import glob, json, os, sys, time, zlib
import numpy as np, pandas as pd, torch

CFG = dict(channels=["VV", "VH"], min_score=float(os.environ.get("XV3_MIN_SCORE", 0.3)), radius_px=30.0, max_tiles_per_scene=150, batch=32)
ROOT = os.environ.get("XV3_DATA", "/kaggle/input")
OUT = os.environ.get("XV3_OUT", "/kaggle/working")
scene_files = sorted(p for p in glob.glob(ROOT + "/**/*.npz", recursive=True)
                     if "_shoreline" not in p and ("maritime-xview3-mine" in p or os.environ.get("XV3_ANY_DIR")))
weight_files = sorted(glob.glob(ROOT + "/**/detector_VV+VH@*.pt", recursive=True))
assert scene_files, "add the maritime-xview3-mine dataset as input"
assert weight_files, "add the xview3_train notebook output as input"
DATA = os.path.dirname(scene_files[0])
with open(f"{OUT}/maritime_detector.py", "w") as f:
    f.write(DETECTOR_SRC)
sys.path.insert(0, OUT)
import maritime_detector as D
dev = "cuda" if torch.cuda.is_available() else "cpu"
models = []
for w in weight_files:
    m = D.build_model("resnet18", pretrained=False, in_ch=len(CFG["channels"]))
    m.load_state_dict(torch.load(w, map_location="cpu"))
    models.append(m.to(dev).eval())
print(f"{len(scene_files)} scenes | {len(models)} models | {dev}")

In [ ]:
lab = pd.read_csv(f"{DATA}/labels.csv")
def as_float(s):
    return pd.to_numeric(s.map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}).where(s.notna()), errors="coerce")
def gt_for(sid):
    g = lab[lab.scene_id == sid]
    shore = pd.to_numeric(g.distance_from_shore_km, errors="coerce") if "distance_from_shore_km" in g else np.nan
    return pd.DataFrame({"row": g.detect_scene_row.astype(float), "col": g.detect_scene_column.astype(float),
                         "is_vessel": as_float(g.is_vessel), "is_fishing": as_float(g.is_fishing),
                         "length_m": pd.to_numeric(g.vessel_length_m, errors="coerce"),
                         "confidence": g.confidence.fillna("").astype(str), "shore_km": shore}).to_records(index=False)
def shoreline(sid):
    p = f"{DATA}/{sid}_shoreline.npy"
    return list(np.load(p, allow_pickle=True)) if os.path.exists(p) else None

## Run the models, collect false alarms, cut tiles
A false alarm here = ensemble score >= 0.3 with no label of any confidence within 30 px (300 m). The
radius is looser than the 200 m scoring radius so that slightly misplaced labels are not mistaken for
empty sea. Up to 150 tiles per scene, highest-scoring false alarms first.

In [ ]:
DETS, GT, SHORE, report = {}, {}, {}, []
for p in scene_files:
    sid = os.path.basename(p)[:-4]
    t0 = time.time()
    img = np.load(p)["image"]
    dets, origins = D.predict_scene(models, img, CFG["channels"], batch=CFG["batch"], device=dev,
                                    num_workers=4 if dev == "cuda" else 0)
    ens = dets[-1]
    g = gt_for(sid); GT[sid] = g; SHORE[sid] = shoreline(sid); DETS[sid] = ens
    fa = D.unmatched_detections(ens, np.column_stack([g["row"], g["col"]]) if len(g) else np.zeros((0, 2)),
                                CFG["min_score"], CFG["radius_px"])
    seed = zlib.crc32(sid.encode())
    ims, orgs, objs = D.hard_negative_tiles(img, fa, g, CFG["max_tiles_per_scene"], seed=seed)
    np.savez_compressed(f"{OUT}/hn_{sid}.npz", images=ims, origins=orgs, objects=objs,
                        scene_shape=np.array(img.shape[1:]))
    near = D.shore_distance_px(fa[:, :2], SHORE[sid]) <= 200 if len(fa) else np.zeros(0, bool)
    report.append(dict(scene=sid, labels=len(g), false_alarms_ge03=len(fa),
                       false_alarms_ge04=int((fa[:, 2] >= 0.4).sum()) if len(fa) else 0,
                       near_shore_share=round(float(near.mean()), 2) if len(near) else None,
                       tiles=len(ims), objects_in_tiles=len(objs), seconds=round(time.time() - t0)))
    print(report[-1], flush=True)
    del img
rep = pd.DataFrame(report)
rep.to_csv(f"{OUT}/mining_report.csv", index=False)
print("\ntotal tiles:", rep.tiles.sum(), "| false alarms >= 0.4:", rep.false_alarms_ge04.sum(),
      "| near-shore share:", round(float(np.nanmean(rep.near_shore_share.astype(float))), 2))

## Calibration view: the current models on these scenes
These scenes were trained on (as tiles), so the scores are optimistic; the point is the *shape* of the
threshold curve on whole scenes. The real calibration for the retrained model happens in the eval
notebook.

In [ ]:
def to_rec(d, thr):
    d = d[d[:, 2] >= thr]
    return pd.DataFrame({"row": d[:, 0], "col": d[:, 1], "score": d[:, 2], "length_m": d[:, 3],
                         "is_vessel": d[:, 4], "is_fishing": d[:, 5]}).to_records(index=False)
sweep = pd.DataFrame([dict(threshold=t, **D.xview3_metrics({s: to_rec(d, t) for s, d in DETS.items()}, GT, SHORE))
                      for t in [0.3, 0.4, 0.5, 0.6, 0.7]])
print(sweep[["threshold", "precision", "recall", "f1_detect", "f1_shore", "aggregate", "fp"]].round(3).to_string(index=False))
sweep.to_csv(f"{OUT}/calibration_sweep_v5.csv", index=False)

## What the false alarms look like (top 24 by score, VH)

In [ ]:
import matplotlib.pyplot as plt
top = []
for s, d in DETS.items():
    g = GT[s]
    fa = D.unmatched_detections(d, np.column_stack([g["row"], g["col"]]) if len(g) else np.zeros((0, 2)), 0.4, 30)
    top += [(s, r, c, sc) for r, c, sc in fa[:, :3]]
top = sorted(top, key=lambda t: -t[3])[:24]
fig, axes = plt.subplots(3, 8, figsize=(16, 6.6))
cache = {}
for ax, (s, r, c, sc) in zip(axes.flat, top):
    if s not in cache:
        cache = {s: np.load(f"{DATA}/{s}.npz")["image"]}
    r, c = int(r), int(c)
    ax.imshow(cache[s][1, max(r - 32, 0):r + 32, max(c - 32, 0):c + 32], cmap="gray", vmin=100, vmax=230)
    ax.set_title(f"{sc:.2f}", fontsize=8)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/false_alarms.png", dpi=70)